# 코드2 — 판별기 (이익 계산기)

새 구조의 두 번째 블록. **후보 하나를 받아 이익을 계산하고 승인/반려를 판정한다.**

## 이 코드가 하는 일

1. 코드1이 만든 `CustomerSimulator`를 불러온다
2. **자체 재고 엔진** — 로트 단위 입고·소진·감모·폐기를 직접 굴린다
3. **판별기** — 정책을 받아 매출·원가·폐기손실·이익을 계산
4. **베이스라인 3종 비교** — 무할인 / 룰 기반 마감할인 / 사후 오라클

## 왜 재고를 직접 굴리는가

원천데이터의 판매·폐기 기록을 그대로 쓰면 **가격을 바꿔도 폐기가 안 바뀐다.**
할인을 걸어 더 팔리면 재고가 일찍 소진되고 폐기가 줄어드는 것이 핵심인데,
그 연쇄가 끊기면 정책 비교 자체가 무의미해진다.

그래서 **입고(발주)만 외생으로 받고, 판매와 폐기는 우리가 계산한다.**
발주는 우리 의사결정이 아니므로 원천데이터를 그대로 쓰는 것이 맞다.

## 이익 정의

```
이익 = 매출 − 매출원가 − 폐기손실
매출     = Σ 판매수량 × 정가 × (1 − 할인율)
매출원가 = Σ 판매수량 × 원가
폐기손실 = Σ 폐기수량 × 원가  +  폐기 처리비(147,000원/톤)
```

## 임계점

**현행 룰 기반 마감할인의 이익**을 기준으로 잡는다.
우리가 실제로 대체하는 대상이 그것이기 때문이다.
무할인은 하한선, 사후 오라클은 이론적 상한으로 함께 표시한다.

## 0. 환경 설정

구글 드라이브의 `freshwatch` 폴더를 사용한다.

```
내 드라이브 / freshwatch /
├── data /   원천 CSV 7개
├── out /    코드1 산출물 3개
└── out2 /   코드2 산출물 (자동 생성)
```

| 위치 | 파일 |
|---|---|
| `data/` | `customer_v2.csv`, `product.csv`, `store.csv`, `calendar.csv`, `store_calendar.csv`, `store_visitor_profile.csv`, `inventory.csv` |
| `out/` | `customer_sim.py`, `sim_arrays.npz`, `params_customer_sim.json` |

파일이 하나라도 없으면 아래 셀이 **어떤 파일이 없는지 알려주고 즉시 중단**한다.
드라이브에 넣고 다시 실행하면 된다.

In [ ]:
# ── 구글 드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')

import os
BASE = '/content/drive/MyDrive/freshwatch'
DATA = os.path.join(BASE, 'data')     # 원천 CSV 7개
OUT1 = os.path.join(BASE, 'out')      # 코드1 산출물
OUT  = os.path.join(BASE, 'out2')     # 코드2 산출물
os.makedirs(OUT, exist_ok=True)

# 필요한 파일이 다 있는지만 확인한다 (없으면 즉시 중단)
NEED_CSV = ['customer_v2.csv', 'product.csv', 'store.csv', 'calendar.csv',
            'store_calendar.csv', 'store_visitor_profile.csv', 'inventory.csv']
NEED_OUT1 = ['customer_sim.py', 'sim_arrays.npz', 'params_customer_sim.json']

m1 = [n for n in NEED_CSV if not os.path.exists(os.path.join(DATA, n))]
m2 = [n for n in NEED_OUT1 if not os.path.exists(os.path.join(OUT1, n))]
if m1 or m2:
    if m1: print('freshwatch/data/ 에 없는 파일 :', ', '.join(m1))
    if m2: print('freshwatch/out/  에 없는 파일 :', ', '.join(m2))
    raise FileNotFoundError('드라이브에 위 파일을 넣고 다시 실행하세요.')

print('DATA =', DATA)
print('OUT1 =', OUT1, '(코드1 산출물)')
print('OUT  =', OUT)
print()
print('파일 확인 완료 — 원천 CSV 7개 · 코드1 산출물 3개')

In [ ]:
import numpy as np, pandas as pd, json, time, sys, copy

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 50)
RNG = np.random.default_rng(20260805)

CATS = ['produce', 'meat', 'dairy', 'deli', 'cheese']
CAT_KR = {'produce': '과채', 'meat': '축산', 'dairy': '유제품', 'deli': '델리', 'cheese': '치즈'}
SLOTS = ['morning', 'lunch', 'afternoon', 'evening', 'closing']

DISPOSAL_FEE_PER_KG = 147.0     # 환경부 고시 147,000원/톤
SHRINKAGE_RATE = 0.0255         # 원천데이터 실측 일일 감모율
print('numpy', np.__version__, '| pandas', pd.__version__)

## 1. 코드1 산출물 불러오기

`customer_sim.py`, `sim_arrays.npz`, `params_customer_sim.json` 세 개가 필요하다.
코드1을 먼저 실행해 `out/` 폴더에 만들어 두어야 한다.

In [ ]:
sys.path.insert(0, OUT1)
import customer_sim
import importlib; importlib.reload(customer_sim)
from customer_sim import CustomerSimulator

arr = np.load(f'{OUT1}/sim_arrays.npz', allow_pickle=True)
PS, FS, BSF, LAM, NSEG = arr['PS'], arr['FS'], arr['BSF'], arr['LAM'], arr['NSEG']
PREF_J, TRIP_BUD = arr['PREF_J'], arr['TRIP_BUD']
CAT_IDX, BASE_PRICE, BASE_COST = arr['CAT_IDX'], arr['BASE_PRICE'], arr['BASE_COST']
PID = arr['PID']
P, S = len(PID), len(PS)

with open(f'{OUT1}/params_customer_sim.json', encoding='utf-8') as f:
    saved = json.load(f)
PARAMS = dict(alpha=np.array(saved['alpha']), c=saved['c'],
              beta_disc=saved['beta_disc'], beta_fresh=saved['beta_fresh'],
              beta_bud=saved['beta_bud'], gamma=saved['gamma'])
VSCALE, EQ = saved['visit_scale'], saved['EQ']

print('세그먼트', S, '· 상품', P)
print('파라미터', {k: (np.round(v, 3).tolist() if isinstance(v, np.ndarray) else round(v, 4))
                 for k, v in PARAMS.items()})

In [ ]:
prod  = pd.read_csv(f'{DATA}/product.csv')
store = pd.read_csv(f'{DATA}/store.csv')
cal   = pd.read_csv(f'{DATA}/calendar.csv')
scal  = pd.read_csv(f'{DATA}/store_calendar.csv')
vprof = pd.read_csv(f'{DATA}/store_visitor_profile.csv')
inv   = pd.read_csv(f'{DATA}/inventory.csv')
for d in (prod, store, cal, scal, vprof):
    d.columns = [c.lstrip('\ufeff') for c in d.columns]

inv['current_date'] = pd.to_datetime(inv['current_date'])
inv['expiry_date'] = pd.to_datetime(inv['expiry_date'])
cal['date'] = pd.to_datetime(cal['date'])
scal['date'] = pd.to_datetime(scal['date'])

prod = prod.set_index('product_id').loc[list(PID)].reset_index()
WEIGHT = prod.standard_weight_kg.values.astype(float)
MAXDISC = prod.max_discount_rate.values.astype(float)
IDX_P = {p: i for i, p in enumerate(PID)}
CATNAME = prod.category.values
print('상품 정렬 확인:', (prod.product_id.values == PID).all())
print('정가 범위 %d~%d원 · 원가율 평균 %.3f' %
      (BASE_PRICE.min(), BASE_PRICE.max(), (BASE_COST / BASE_PRICE).mean()))

## 2. 시뮬레이터 재구성

코드1과 동일한 방문객·달력 룩업을 만든다.
다만 **재고 상태는 우리가 직접 관리**하므로 `inv_by`는 비워 두고,
`expected_demand` 호출 시 신선도와 가용재고를 인자로 직접 넘긴다.

In [ ]:
store_w = store.set_index('store_id').floating_idx
store_w = store_w / store_w.sum()
STORES = list(store_w.index)

DOW_F = {'MON': .88, 'TUE': .88, 'WED': .90, 'THU': .93,
         'FRI': 1.08, 'SAT': 1.32, 'SUN': 1.25}
day = cal[['date', 'day_of_week', 'day_type', 'season_index', 'event_index']].copy()
day['f_day'] = (day.day_of_week.map(DOW_F) * day.event_index.fillna(1.0)
                * day.season_index.fillna(1.0))
day_info = {r.date: (r.day_type, r.f_day) for _, r in day.iterrows()}
open_map = scal.set_index(['date', 'store_id']).is_open.to_dict()

vp = vprof.merge(store[['store_id', 'area_type', 'close_hour']], on=['area_type', 'close_hour'])
slot_ratio = {(r.store_id, r.day_type, r.time_slot): r.visitor_ratio for _, r in vp.iterrows()}
slot_hours = {}
for _, r in vp.drop_duplicates(['store_id', 'time_slot']).iterrows():
    slot_hours.setdefault(r.store_id, {})[r.time_slot] = list(range(int(r.start_hour), int(r.end_hour)))
STORE_HOURS = {s: sorted({h for hs in slot_hours[s].values() for h in hs}) for s in STORES}

seg_arrays = (PS, FS, BSF, LAM, NSEG, PREF_J, TRIP_BUD, CAT_IDX, BASE_PRICE)
lookups = (store_w, slot_ratio, slot_hours, open_map, day_info, {}, IDX_P)
sim = CustomerSimulator(PARAMS, VSCALE, EQ, seg_arrays, lookups)

for s in STORES:
    print(s, '영업시간', STORE_HOURS[s][0], '~', STORE_HOURS[s][-1], f'({len(STORE_HOURS[s])}시간)')

## 3. 재고 엔진

원천데이터에서 **로트 입고 일정만** 뽑아 온다. 이후 소진·감모·폐기는 직접 계산한다.

- 판매는 **유통기한이 임박한 로트부터**(FIFO) 차감
- 하루 끝에 잔여재고의 2.55%를 감모 처리 (원천데이터 실측)
- 유통기한이 지나면 잔여 전량 폐기

In [ ]:
lots_src = (inv.sort_values('current_date')
              .groupby(['store_id', 'product_id', 'lot_id'])
              .agg(arrival=('current_date', 'first'), qty=('inbound_qty', 'first'),
                   expiry=('expiry_date', 'first'), cost=('unit_cost', 'first'),
                   price=('unit_price', 'first'))
              .reset_index())
lots_src = lots_src[lots_src.qty > 0].copy()
lots_src['pi'] = lots_src.product_id.map(IDX_P)
print('로트', len(lots_src), '· 총 입고량', int(lots_src.qty.sum()))

# 신선도 룩업 (상품 x 잔여기한) — 원천데이터 평균
FRESH_LUT = (inv.groupby(['product_id', 'days_to_expiry']).freshness_score.mean()
               .reset_index())
FRESH_LUT['pi'] = FRESH_LUT.product_id.map(IDX_P)
FRESH_MAP = {(int(r.pi), int(r.days_to_expiry)): float(r.freshness_score)
             for _, r in FRESH_LUT.iterrows()}
FRESH_DEFAULT = float(inv.freshness_score.mean())


def freshness(pi, dte):
    return FRESH_MAP.get((pi, int(dte)), FRESH_DEFAULT)


print('신선도 룩업', len(FRESH_MAP), '조합 · 기본값', round(FRESH_DEFAULT, 3))

## 3.5 잔여기한별 차등 가격 — ESL의 실제 동작

ESL(전자가격표시기)은 **같은 상품이라도 잔여 유통기한이 다르면 다른 가격을 표시할 수 있다.**
예를 들어 오후 2시에 우유 매대는 다음과 같이 나뉜다.

| 잔여기한 | 표시 가격 |
|---|---|
| D-3 이상 | 15% 할인 |
| D-2 | 20% 할인 |
| D-1 | 25% 할인 |
| 당일 만료 | 40% 할인 |

이것이 이 시스템의 핵심이다. **폐기될 물량에만 깊은 할인을 걸고, 신선한 재고는 정가에 가깝게 유지**할 수 있다.

### 모형에 반영하는 방법

같은 상품의 서로 다른 잔여기한 재고는 **경쟁하는 대안(substitute)** 이다.
소비자는 "싸지만 임박한 것"과 "비싸지만 신선한 것" 중에서 고른다.
그래서 **중첩 로짓(nested logit)** 구조를 쓴다.

```
1단계  이 상품을 살까?      P(상품) = sigmoid( logsumexp_b V(b) )
2단계  어느 잔여기한을 살까?  share(b) = softmax_b V(b)

기대 판매수량(상품, 버킷) = 방문객 × P(상품) × share(버킷) × 평균 구매수량
```

`logsumexp`를 쓰면 **선택지가 많고 매력적일수록 그 상품을 살 확률 자체가 올라간다.**
임박 재고에 할인을 걸면 그 상품의 총 수요가 늘고, 동시에 수요가 할인된 재고 쪽으로 쏠린다.
현장에서 마감 스티커가 붙은 것부터 팔리는 현상과 일치한다.

버킷이 하나뿐이면 `logsumexp`가 그 값 자체가 되므로, 기존 단일 가격 모형과 동일해진다.

In [ ]:
# ── 잔여기한 버킷 정의
DTE_EDGES = [0, 1, 2, 3]          # 당일만료 / D-1 / D-2 / D-3 이상
DTE_LABEL = ['당일만료', 'D-1', 'D-2', 'D-3 이상']
NB = len(DTE_EDGES)


def dte_bucket(d):
    """잔여기한(일) -> 버킷 인덱스"""
    if d <= 0:
        return 0
    return 3 if d >= 3 else int(d)


# ── 효용 계산 (세그먼트 x 상품 x 버킷) — 한 번의 벡터 연산
_ALPHA_P = None          # 재보정 후 상품 단위 alpha


def utilities_dte(disc_mat, fresh_mat):
    """disc_mat, fresh_mat: (P x B) -> U: (S x P x B)"""
    pr = sim.p
    price = BASE_PRICE[:, None] * (1.0 - disc_mat)                 # (P x B)
    U = (pr['alpha'][sim.CAT_IDX][None, :, None]
         + pr['gamma'] * PREF_J[:, :, None]
         + pr['beta_disc'] * PS[:, None, None] * disc_mat[None, :, :]
         + pr['beta_fresh'] * FS[:, None, None] * (fresh_mat[None, :, :] - 0.6)
         - pr['beta_bud'] * PS[:, None, None]
           * np.log(price[None, :, :] / TRIP_BUD[:, None, None])
         + np.log(BSF)[:, None, None]
         + pr['c'])
    return U


def demand_by_dte(store_id, date, hour, disc_mat, avail_mat, fresh_mat):
    """중첩 로짓으로 (P x B) 기대 판매수량 계산"""
    V = sim.visitors(store_id, date, hour)
    if V is None:
        return np.zeros((P, NB))
    U = utilities_dte(disc_mat, fresh_mat)
    U = np.where((avail_mat > 0)[None, :, :], U, -1e9)             # 재고 없는 버킷 제외
    m = U.max(axis=2, keepdims=True)
    e = np.exp(U - m)
    tot = e.sum(axis=2, keepdims=True)
    IV = (m + np.log(np.maximum(tot, 1e-300)))[:, :, 0]            # (S x P) 포괄가치
    Ptot = 1.0 / (1.0 + np.exp(-np.clip(IV, -60, 60)))                               # 상품 구매확률
    share = e / np.maximum(tot, 1e-300)                            # 버킷 선택 비중
    Pmat = Ptot[:, :, None] * share                                # (S x P x B)
    return np.einsum('s,spb->pb', V, Pmat) * EQ


# ── 시간대 구간: 동적 가격의 핵심 축
# 마감이 가까울수록 "그때까지 안 팔린 물량"에만 할인이 걸린다.
TIME_LABEL = ['개점~마감3시간전', '마감 3~1시간전', '마감 1시간']
NT = len(TIME_LABEL)


def time_band(store_id, hour):
    close = STORE_HOURS[store_id][-1]
    if hour >= close:
        return 2
    if hour >= close - 2:
        return 1
    return 0


print('잔여기한 버킷', NB, '개:', DTE_LABEL)
print('시간대 구간', NT, '개:', TIME_LABEL)

In [ ]:
class InventoryEngine:
    """로트 단위 재고. 입고는 외생, 소진·감모·폐기는 내생."""

    def __init__(self, lots_df, store_id):
        d = lots_df[lots_df.store_id == store_id]
        self.arrivals = {k: v[['pi', 'qty', 'expiry', 'cost', 'price']].values
                         for k, v in d.groupby('arrival')}
        self.reset()

    def reset(self):
        # 로트: [pi, 잔여수량, 만료일, 원가, 정가]
        self.lots = []
        self.log = dict(sold=0.0, revenue=0.0, cogs=0.0,
                        waste_qty=0.0, waste_cost=0.0, disposal_fee=0.0,
                        expired_qty=0.0, shrink_qty=0.0)

    def clone(self):
        """입고 일정은 공유하고 로트 상태만 복제. 예열 재실행을 피한다."""
        e = InventoryEngine.__new__(InventoryEngine)
        e.arrivals = self.arrivals
        e.lots = [list(l) for l in self.lots]
        e.log = dict(self.log)
        return e

    def receive(self, date):
        for row in self.arrivals.get(date, []):
            self.lots.append([int(row[0]), float(row[1]), row[2], float(row[3]), float(row[4])])

    def state(self, date):
        """(가용재고 P, 신선도 P) — 상품 단위 합산 (참고용)"""
        a, f = self.state_by_dte(date)
        avail = a.sum(1)
        fw = (a * f).sum(1)
        fresh = np.where(avail > 0, fw / np.maximum(avail, 1e-9), FRESH_DEFAULT)
        return avail, fresh

    def state_by_dte(self, date):
        """(가용재고 P x B, 신선도 P x B) — 잔여기한 버킷별.
        ESL은 같은 상품이라도 잔여기한이 다르면 다른 가격을 표시할 수 있다."""
        avail = np.zeros((P, NB)); fw = np.zeros((P, NB))
        for lot in self.lots:
            if lot[1] <= 0:
                continue
            b = dte_bucket((lot[2] - date).days)
            avail[lot[0], b] += lot[1]
            fw[lot[0], b] += lot[1] * freshness(lot[0], (lot[2] - date).days)
        fresh = np.where(avail > 0, fw / np.maximum(avail, 1e-9), FRESH_DEFAULT)
        return avail, fresh

    def min_dte(self, date):
        """상품별 가장 임박한 로트의 잔여기한 (참고용, 없으면 99)"""
        out = np.full(P, 99)
        for lot in self.lots:
            if lot[1] <= 0:
                continue
            d = (lot[2] - date).days
            if d < out[lot[0]]:
                out[lot[0]] = d
        return out

    def sell(self, qty_mat, disc_mat, date):
        """잔여기한 버킷별 판매. qty_mat, disc_mat 모두 (P x B).
        같은 버킷 안에서는 만료 임박 순으로 차감한다."""
        sold = np.zeros((P, NB))
        need = qty_mat.copy()
        order = sorted(range(len(self.lots)), key=lambda i: self.lots[i][2])
        for i in order:
            lot = self.lots[i]
            if lot[1] <= 0:
                continue
            pi = lot[0]
            b = dte_bucket((lot[2] - date).days)
            if need[pi, b] <= 1e-9:
                continue
            take = min(lot[1], need[pi, b])
            lot[1] -= take; need[pi, b] -= take; sold[pi, b] += take
            self.log['revenue'] += take * lot[4] * (1.0 - disc_mat[pi, b])
            self.log['cogs'] += take * lot[3]
        self.log['sold'] += sold.sum()
        return sold

    def end_of_day(self, date):
        """감모 + 만료 폐기"""
        for lot in self.lots:
            if lot[1] <= 0:
                continue
            sh = lot[1] * SHRINKAGE_RATE
            lot[1] -= sh
            self.log['shrink_qty'] += sh
            self.log['waste_qty'] += sh
            self.log['waste_cost'] += sh * lot[3]
            self.log['disposal_fee'] += sh * WEIGHT[lot[0]] * DISPOSAL_FEE_PER_KG
        keep = []
        for lot in self.lots:
            if (lot[2] - date).days <= 0:
                if lot[1] > 0:
                    self.log['expired_qty'] += lot[1]
                    self.log['waste_qty'] += lot[1]
                    self.log['waste_cost'] += lot[1] * lot[3]
                    self.log['disposal_fee'] += lot[1] * WEIGHT[lot[0]] * DISPOSAL_FEE_PER_KG
            elif lot[1] > 1e-9:
                keep.append(lot)
        self.lots = keep


print('InventoryEngine 정의 완료')

## 4. 판별기

정책 함수를 받아 평가 기간 전체를 굴린다.
정책 함수의 서명은 다음과 같다.

```
policy(store_id, date, hour, avail_mat) -> (38 x 4) 할인율 행렬
```

행은 상품, 열은 잔여기한 버킷(당일만료 / D-1 / D-2 / D-3 이상)이다.
**ESL이 잔여기한별로 다른 가격을 표시할 수 있으므로** 같은 상품이라도 버킷마다 할인율이 다르다.

이 서명이 중요하다. **후보 생성기(코드3)가 만드는 것도 정확히 이 함수**다.
판별기는 정책의 내부 구조를 알 필요가 없다.

In [ ]:
WARMUP_START = pd.Timestamp('2025-11-15')     # 재고를 채우기 위한 예열 구간
EVAL_START   = pd.Timestamp('2025-12-01')
EVAL_END     = pd.Timestamp('2025-12-31')
EVAL_DAYS = pd.date_range(WARMUP_START, EVAL_END, freq='D')
print('예열 %s ~ %s · 평가 %s ~ %s (%d일)'
      % (WARMUP_START.date(), (EVAL_START - pd.Timedelta(days=1)).date(),
         EVAL_START.date(), EVAL_END.date(), (EVAL_END - EVAL_START).days + 1))


def evaluate(policy, engines=None, days=None):
    """정책 -> 이익 및 부가 지표.

    정책 서명: policy(store_id, date, hour, avail_mat) -> (P x B) 할인율
    P는 상품, B는 잔여기한 버킷이다. ESL이 잔여기한별 차등 가격을 표시할 수 있으므로
    같은 상품이라도 버킷마다 다른 할인율을 지정한다.
    """
    engines = engines or {s: InventoryEngine(lots_src, s) for s in STORES}
    days = EVAL_DAYS if days is None else days
    for date in days:
        in_eval = date >= EVAL_START
        for s in STORES:
            eng = engines[s]
            eng.receive(date)
            if open_map.get((date, s), 0) == 1:
                pol = policy if in_eval else rule_policy
                for h in STORE_HOURS[s]:
                    avail, fresh = eng.state_by_dte(date)
                    if avail.sum() <= 0:
                        continue
                    disc = np.clip(pol(s, date, h, avail), 0.0, MAXDISC[:, None])
                    dem = demand_by_dte(s, date, h, disc, avail, fresh)
                    eng.sell(np.minimum(dem, avail), disc, date)
            eng.end_of_day(date)
            if date == EVAL_START - pd.Timedelta(days=1):
                eng.log = dict.fromkeys(eng.log, 0.0)      # 예열 기록 초기화

    agg = {k: sum(engines[s].log[k] for s in STORES) for k in engines[STORES[0]].log}
    agg['profit'] = agg['revenue'] - agg['cogs'] - agg['waste_cost'] - agg['disposal_fee']
    agg['waste_rate'] = agg['waste_qty'] / max(agg['sold'] + agg['waste_qty'], 1e-9)
    agg['margin'] = agg['profit'] / max(agg['revenue'], 1e-9)
    return agg

## 4.5 판별기 캘리브레이션 — 상품 단위 재보정

코드1의 캘리브레이션은 **카테고리 단위**였다. 카테고리 안에서 어떤 상품이 팔리는지는
맞추지 않았다. 원천 재고를 그대로 읽을 때는 문제가 없었지만,
**재고를 직접 굴리는 판별기에서는 치명적**이다.

예를 들어 축산 카테고리 비중이 맞아도, 그 안에서 한우 등심 대신 닭가슴살만 팔리면
한우는 계속 입고되어 쌓이다가 전량 폐기된다. 한우 원가가 65,000원이므로
폐기손실이 폭발하고 이익이 음수가 된다.

그래서 **상품 단위 기본 인기도**로 재보정한다.

- 목표: 현행 룰 기반 정책으로 12월을 굴렸을 때 상품별 판매수량이 원천 실적과 일치
- 방법: 상품별 IPF (판매 부족한 상품은 인기도를 올리고, 과다한 상품은 내림)
- 이는 **판별기 환경에서 현행 정책이 실제 실적을 재현하는지 확인하는 절차**이기도 하다

재보정 후 상품 단위 인기도는 정적 파라미터로 고정되고, 이후 정책 비교에 동일하게 쓰인다.

In [ ]:
# 현행 룰 기반 정책 (재보정과 베이스라인에서 공통 사용)
# 잔여기한 버킷별 할인율: 당일만료 40% / D-1 30% / D-2 20% / D-3 이상 0%
RULE_VEC = np.array([0.40, 0.30, 0.20, 0.00])


def rule_policy(store_id, date, hour, avail_mat):
    return np.tile(RULE_VEC, (P, 1))


def no_discount(store_id, date, hour, avail_mat):
    return np.zeros((P, NB))


# 목표: 원천데이터 12월 상품별 판매수량
_dec = inv[(inv.current_date >= EVAL_START) & (inv.current_date <= EVAL_END)]
tgt_prod = np.zeros(P)
for pid, v in _dec.groupby('product_id').daily_sold_qty.sum().items():
    if pid in IDX_P:
        tgt_prod[IDX_P[pid]] = v
TGT_TOTAL = tgt_prod.sum()
print('원천 12월 판매수량 %d개 · 상품별 최대 %d / 최소 %d'
      % (TGT_TOTAL, tgt_prod.max(), tgt_prod.min()))

# alpha를 카테고리 5개 -> 상품 38개로 확장
alpha_prod = PARAMS['alpha'][CAT_IDX].astype(float).copy()
sim.CAT_IDX = np.arange(P)          # 상품 단위 인덱싱으로 전환
sim.p = dict(PARAMS); sim.p['alpha'] = alpha_prod
print('상품 단위 인기도로 전환 · 초기 alpha 범위 %.3f ~ %.3f'
      % (alpha_prod.min(), alpha_prod.max()))

In [ ]:
def sold_by_product(policy):
    """정책 하에서 상품별 판매수량"""
    engines = {st: InventoryEngine(lots_src, st) for st in STORES}
    sold = np.zeros(P)
    for date in EVAL_DAYS:
        in_eval = date >= EVAL_START
        for st in STORES:
            eng = engines[st]
            eng.receive(date)
            if open_map.get((date, st), 0) == 1:
                pol = policy if in_eval else rule_policy
                for h in STORE_HOURS[st]:
                    avail, fresh = eng.state_by_dte(date)
                    if avail.sum() <= 0:
                        continue
                    disc = np.clip(pol(st, date, h, avail), 0.0, MAXDISC[:, None])
                    dem = demand_by_dte(st, date, h, disc, avail, fresh)
                    s_ = eng.sell(np.minimum(dem, avail), disc, date)
                    if in_eval:
                        sold += s_.sum(1)
            eng.end_of_day(date)
    return sold


t0 = time.time()
for it in range(12):
    sold = sold_by_product(rule_policy)
    tot_err = sold.sum() / TGT_TOTAL - 1
    sim.p['c'] = sim.p['c'] - 0.55 * np.log(max(sold.sum(), 1) / TGT_TOTAL)
    sh_now = sold / max(sold.sum(), 1e-9)
    sh_tgt = tgt_prod / TGT_TOTAL
    step = np.log(np.maximum(sh_tgt, 1e-5) / np.maximum(sh_now, 1e-5))
    sim.p['alpha'] = sim.p['alpha'] + 0.6 * np.clip(step, -1.5, 1.5)
    sim.p['alpha'] -= sim.p['alpha'].mean()
    l1 = float(np.abs(sh_now - sh_tgt).sum())
    print('[%2d] 총량 %6.0f (오차 %+6.2f%%) · 상품비중 L1 %.4f' % (it, sold.sum(), tot_err * 100, l1))
    if abs(tot_err) < 0.02 and l1 < 0.06:
        break
print('재보정 %.1f분' % ((time.time() - t0) / 60))
PARAMS_CAL = dict(sim.p)

In [ ]:
# 재보정 결과 확인 — 상품별 판매수량 대조
sold_fin = sold_by_product(rule_policy)
chk = pd.DataFrame({'상품': prod.product_name.values,
                    '카테고리': [CAT_KR[c] for c in prod.category.values],
                    '원천': tgt_prod, '시뮬': sold_fin.round(0)})
chk['오차'] = (chk.시뮬 - chk.원천)
chk['비중_원천'] = (chk.원천 / chk.원천.sum() * 100).round(2)
chk['비중_시뮬'] = (chk.시뮬 / chk.시뮬.sum() * 100).round(2)
PROD_CHECK = chk
print('총량 오차 %+.2f%% · 상품비중 L1 %.4f'
      % ((sold_fin.sum() / TGT_TOTAL - 1) * 100,
         np.abs(sold_fin / sold_fin.sum() - tgt_prod / TGT_TOTAL).sum()))
print()
print(chk.reindex(chk.원천.sort_values(ascending=False).index)
        .head(10)[['상품', '카테고리', '원천', '시뮬', '비중_원천', '비중_시뮬']].to_string(index=False))

## 4.6 고속 판별기 — 예열 구간 캐싱

예열 구간(11/15~11/30)은 어떤 정책을 평가하든 동일하게 현행 룰로 굴린다.
따라서 한 번만 계산해두고 로트 상태만 복제하면 평가 시간을 크게 줄일 수 있다.
이후의 오라클 탐색과 대리모델 학습 데이터 생성이 모두 이 고속 경로를 쓴다.

In [ ]:
# ── 예열 구간은 어떤 정책이든 동일(룰 기반)하므로 한 번만 계산해 재사용한다
def build_warm_state():
    engines = {s: InventoryEngine(lots_src, s) for s in STORES}
    for date in pd.date_range(WARMUP_START, EVAL_START - pd.Timedelta(days=1)):
        for s in STORES:
            eng = engines[s]
            eng.receive(date)
            if open_map.get((date, s), 0) == 1:
                for h in STORE_HOURS[s]:
                    avail, fresh = eng.state_by_dte(date)
                    if avail.sum() <= 0:
                        continue
                    disc = rule_policy(s, date, h, avail)
                    dem = demand_by_dte(s, date, h, disc, avail, fresh)
                    eng.sell(np.minimum(dem, avail), disc, date)
            eng.end_of_day(date)
    for s in STORES:
        engines[s].log = dict.fromkeys(engines[s].log, 0.0)
    return engines


WARM = build_warm_state()
EVAL_ONLY = pd.date_range(EVAL_START, EVAL_END, freq='D')


def evaluate_fast(policy):
    """예열 상태를 재사용하는 고속 판별기. evaluate()와 결과가 동일하다."""
    engines = {s: WARM[s].clone() for s in STORES}
    for date in EVAL_ONLY:
        for s in STORES:
            eng = engines[s]
            eng.receive(date)
            if open_map.get((date, s), 0) == 1:
                for h in STORE_HOURS[s]:
                    avail, fresh = eng.state_by_dte(date)
                    if avail.sum() <= 0:
                        continue
                    disc = np.clip(policy(s, date, h, avail), 0.0, MAXDISC[:, None])
                    dem = demand_by_dte(s, date, h, disc, avail, fresh)
                    eng.sell(np.minimum(dem, avail), disc, date)
            eng.end_of_day(date)
    agg = {k: sum(engines[s].log[k] for s in STORES) for k in engines[STORES[0]].log}
    agg['profit'] = agg['revenue'] - agg['cogs'] - agg['waste_cost'] - agg['disposal_fee']
    agg['waste_rate'] = agg['waste_qty'] / max(agg['sold'] + agg['waste_qty'], 1e-9)
    agg['margin'] = agg['profit'] / max(agg['revenue'], 1e-9)
    return agg


t0 = time.time(); a = evaluate(rule_policy); t_slow = time.time() - t0
t0 = time.time(); b = evaluate_fast(rule_policy); t_fast = time.time() - t0
print('기존 %.2f초 → 고속 %.2f초 (%.1f배)' % (t_slow, t_fast, t_slow / max(t_fast, 1e-9)))
print('이익 차이 %.6f%%  (0이면 동일)' % (abs(b['profit'] / a['profit'] - 1) * 100))

## 5. 베이스라인 3종

| 정책 | 내용 | 역할 |
|---|---|---|
| 무할인 | 전 상품·전 시간 0% | 하한선 |
| 룰 기반 마감할인 | 당일만료 40% / D-1 30% / D-2 20% (하루 종일 동일) | 현행 방식 |
| 사후 오라클 | 잔여기한 × 시간대 격자에서 좌표상승법으로 최적화 | 이론적 상한 |

### 시간대 축이 왜 필요한가

현행 룰은 **하루 종일 같은 할인율**을 건다.
그러면 당일 만료 재고 중 "어차피 팔릴 물량"까지 아침부터 깎게 된다.

실제로 필요한 것은 **마감이 가까워졌을 때 그때까지 안 팔린 물량에만** 할인을 거는 것이다.
ESL이 있으면 이것이 가능하고, 그것이 이 시스템의 존재 이유다.

그래서 오라클의 탐색 공간을 **잔여기한 4 × 시간대 3 = 12개 좌표**로 둔다.

| 축 | 구분 |
|---|---|
| 잔여기한 | 당일만료 / D-1 / D-2 / D-3 이상 |
| 시간대 | 개점~마감 3시간 전 / 마감 3~1시간 전 / 마감 1시간 |

오라클이 찾은 값이 **"언제, 얼마나 임박한 재고에, 얼마를 깎아야 하는가"** 의 답이 된다.

In [ ]:
t0 = time.time()
R_NONE = evaluate_fast(no_discount)
print('무할인 완료 %.1f초' % (time.time() - t0))
t0 = time.time()
R_RULE = evaluate_fast(rule_policy)
print('룰 기반 완료 %.1f초' % (time.time() - t0))

for nm, r in [('무할인', R_NONE), ('룰 기반', R_RULE)]:
    print('%-8s 매출 %10.0f · 이익 %9.0f · 폐기율 %5.2f%% · 마진 %5.2f%%'
          % (nm, r['revenue'], r['profit'], r['waste_rate'] * 100, r['margin'] * 100))

# 판별기 타당성 검증 — 현행 정책이 원천 실적을 재현하는가
SRC_WASTE_RATE = float(_dec.daily_waste_qty.sum()
                       / (_dec.daily_sold_qty.sum() + _dec.daily_waste_qty.sum()))
print()
print('[판별기 검증] 현행 룰 기반 정책 하에서')
print('  판매수량  원천 %5.0f개 vs 시뮬 %5.0f개 (오차 %+.2f%%)'
      % (TGT_TOTAL, R_RULE['sold'], (R_RULE['sold'] / TGT_TOTAL - 1) * 100))
print('  폐기율    원천 %.2f%% vs 시뮬 %.2f%%'
      % (SRC_WASTE_RATE * 100, R_RULE['waste_rate'] * 100))

In [ ]:
# 사후 오라클 — 잔여기한 x 시간대 격자 좌표상승법
#
# 시간대 축이 핵심이다. 하루 종일 같은 할인율을 걸면
# "어차피 팔릴 물량"까지 아침부터 깎게 된다.
# 마감이 가까울수록 그때까지 안 팔린 물량에만 할인이 걸리므로,
# 시간대를 나누면 할인의 표적이 훨씬 정확해진다.

DTE_BUCKETS = list(range(NB))
GRID = np.round(np.arange(0, 0.41, 0.10), 2)   # 0/10/20/30/40%
COORDS = [(b, t) for b in range(NB) for t in range(NT)]   # 4 x 3 = 12 좌표
COORD_LABEL = [f'{DTE_LABEL[b]} · {TIME_LABEL[t]}' for b, t in COORDS]


def make_table_policy(table):
    """table[(dte_bucket, time_band)] -> 할인율"""
    M = np.zeros((NT, P, NB))
    for (b, t), v in table.items():
        M[t, :, b] = v

    def pol(store_id, date, hour, avail_mat):
        return M[time_band(store_id, hour)]
    return pol


table = {(b, t): 0.0 for b in range(NB) for t in range(NT)}
best = evaluate_fast(make_table_policy(table))['profit']
print('초기 이익 %.0f (무할인)' % best)

t0 = time.time()
for sweep in range(3):
    improved = False
    for (b, t) in COORDS:
        cur = table[(b, t)]
        for g in GRID:
            if g == cur:
                continue
            table[(b, t)] = g
            v = evaluate_fast(make_table_policy(table))['profit']
            if v > best + 1:
                best, cur, improved = v, g, True
            else:
                table[(b, t)] = cur
        table[(b, t)] = cur
    print('  sweep %d 이익 %.0f (%.1f분)' % (sweep + 1, best, (time.time() - t0) / 60))
    if not improved:
        break

ORACLE_TABLE = dict(table)
R_ORACLE = evaluate_fast(make_table_policy(ORACLE_TABLE))
print('\n오라클 완료 · 이익 %.0f (무할인 대비 %+.2f%%)'
      % (R_ORACLE['profit'], (R_ORACLE['profit'] / R_NONE['profit'] - 1) * 100))
print()
ORACLE_DF = pd.DataFrame([[ORACLE_TABLE[(b, t)] for t in range(NT)] for b in range(NB)],
                         index=DTE_LABEL, columns=TIME_LABEL)
print('최적 할인 테이블 (행=잔여기한, 열=시간대)')
print((ORACLE_DF * 100).astype(int).astype(str) + '%')

## 5.5 손익분기 탄력성 — 이 프로젝트의 핵심 질문

사후 오라클이 **모든 칸에서 0%를 선택**했다면, 그건 버그가 아니라 산수다.

```
정가 마진 19% · 할인 10%p당 판매량 +8.25%

무할인 단위 마진 = 0.19 × 정가
10% 할인 단위 마진 = 0.09 × 정가, 판매량 ×1.0825
              → 0.097 × 정가  (절반 이하)
```

폐기 절감이 이를 상쇄해야 하는데, **할인은 폐기될 물량에만 걸리지 않는다.**
ESL은 상품 단위 단일 가격이므로, 임박 로트가 하나라도 있으면
그 상품의 신선한 재고까지 전부 같은 가격으로 팔린다.

따라서 질문은 "할인을 해야 하나"가 아니라 다음과 같이 바뀐다.

> **할인이 이익이 되려면 가격 탄력성이 얼마여야 하는가?**

`beta_disc`는 캘리브레이션 대상이 아닌 가정값이다.
이 값을 넓은 범위에서 흔들어 **결론이 뒤집히는 지점**을 찾는다.
이것이 합성 데이터의 한계를 정면으로 다루는 방식이다.

In [ ]:
def elasticity_of(beta):
    '''beta_disc -> 할인 10%p당 판매량 변화율'''
    old = sim.p['beta_disc']; sim.p['beta_disc'] = beta
    f_t = np.full(P, 0.7); av = np.ones(P)
    a = sim._probs(np.zeros(P), f_t).sum(1)
    b = sim._probs(np.full(P, 0.40), f_t).sum(1)
    sim.p['beta_disc'] = old
    return float(np.average(b / np.maximum(a, 1e-9) - 1, weights=NSEG) / 4)


def dday_policy(rate):
    """마감 시간대 x 당일만료 재고에만 할인 — 표적이 가장 정확한 형태"""
    M0 = np.zeros((P, NB))
    M1 = np.zeros((P, NB)); M1[:, 0] = rate

    def pol(store_id, date, hour, avail_mat):
        return M1 if time_band(store_id, hour) >= 1 else M0
    return pol


BETA_GRID = [0.0, 0.7, 2.0, 4.0, 7.0, 11.0]
DD_GRID = [0.0, 0.10, 0.20, 0.30, 0.40]

rows = []
t0 = time.time()
beta_bak = sim.p['beta_disc']
for beta in BETA_GRID:
    sim.p['beta_disc'] = beta
    el = elasticity_of(beta)
    base = evaluate_fast(no_discount)['profit']
    best_r, best_p = 0.0, base
    for r in DD_GRID[1:]:
        v = evaluate_fast(dday_policy(r))['profit']
        if v > best_p:
            best_p, best_r = v, r
    rows.append(dict(beta_disc=beta, 탄력성_10pp=el, 무할인이익=base,
                     최적_D_day할인율=best_r, 최적이익=best_p,
                     개선율=best_p / base - 1 if base > 0 else np.nan))
    print('beta %.1f · 탄력성 %+6.2f%%/10%%p · 최적 D-day 할인 %2.0f%% · 개선 %+6.2f%%'
          % (beta, el * 100, best_r * 100, (best_p / base - 1) * 100))
sim.p['beta_disc'] = beta_bak

BREAKEVEN = pd.DataFrame(rows)
BREAKEVEN.to_csv(f'{OUT}/breakeven_elasticity.csv', index=False, encoding='utf-8-sig')
print('\n소요 %.1f분' % ((time.time() - t0) / 60))
BREAKEVEN.round(4)

In [ ]:
# 손익분기점 보간
pos = BREAKEVEN[BREAKEVEN.최적_D_day할인율 > 0]
if len(pos):
    i = pos.index[0]
    if i > 0:
        lo, hi = BREAKEVEN.loc[i-1, '탄력성_10pp'], BREAKEVEN.loc[i, '탄력성_10pp']
        BE_ELAST = (lo + hi) / 2
    else:
        BE_ELAST = BREAKEVEN.loc[i, '탄력성_10pp']
    print('할인이 이익이 되기 시작하는 탄력성: 약 %+.1f%% / 10%%p' % (BE_ELAST * 100))
else:
    BE_ELAST = np.nan
    print('탐색 범위 전체에서 할인이 이익을 개선하지 못함')

CUR_ELAST = elasticity_of(PARAMS_CAL['beta_disc'])
print('현재 가정 탄력성: %+.2f%% / 10%%p' % (CUR_ELAST * 100))
print()
if not np.isnan(BE_ELAST):
    print('해석: 현재 가정(%.1f%%)은 손익분기(%.1f%%)에 미치지 않는다.'
          % (CUR_ELAST * 100, BE_ELAST * 100))
    print('      즉 "할인하지 말라"는 결론은 데이터가 아니라 탄력성 가정에서 나온다.')
    print('      실측 탄력성이 손익분기를 넘는지가 도입 판단의 핵심이다.')

## 5.6 경계 지도 — 언제 할인이 이익인가

판별기의 역할은 "할인하라"도 "하지 말라"도 아니다.
**어떤 조건에서 할인이 이익이고 어떤 조건에서 손해인지, 그 경계를 찾는 것**이다.

경계를 결정하는 축은 둘이다.

| 축 | 의미 | 현재 값 |
|---|---|---|
| **가격 탄력성** | 할인 10%p당 판매량이 얼마나 느는가 | +8.9% <span class="tag">☆ 가정</span> |
| **정가 마진율** | 팔았을 때 얼마가 남는가 | 19.0% (전 품목 20% 고정) |

두 축의 관계를 산수로 보면 이렇다. 마감 시간대에 당일 만료 재고를 할인할 때,

```
폐기될 물량 1개를 팔면        →  +(정가 − 원가·회수) 이득
안 깎아도 팔릴 물량 1개는     →  −(할인폭) 손해

할인이 이익이려면
   (할인으로 늘어난 판매) × 폐기회피 이득  >  (원래 팔릴 물량) × 마진 훼손
```

마진이 얇을수록 훼손이 크고, 탄력성이 낮을수록 추가 판매가 적다.
따라서 **경계는 두 값의 조합으로 정해진다.**

아래에서 두 축을 격자로 훑어 경계를 그린다.
이 지도가 있으면 "우리 상품의 마진과 탄력성이 어디쯤인지"만 확인하면
할인 도입 여부를 바로 판단할 수 있다.

In [ ]:
# ── 마진율 × 탄력성 경계 지도
BASE_COST_ORIG = BASE_COST.copy()
MARGIN_ORIG = float((1 - BASE_COST / BASE_PRICE).mean())

MARGIN_GRID = [0.19, 0.25, 0.30, 0.35]          # 정가 마진율
BETA_GRID2  = [0.7, 2.0, 4.0, 7.0]              # 할인 반응 계수
DD_GRID2    = [0.0, 0.10, 0.20, 0.30, 0.40]

beta_bak = sim.p['beta_disc']
rows, MAP = [], np.zeros((len(MARGIN_GRID), len(BETA_GRID2)))
MAPD = np.zeros_like(MAP)

t0 = time.time()
for mi, m in enumerate(MARGIN_GRID):
    # 마진율만 바꾸고 정가는 유지 (원가를 조정) — 수요는 정가로 결정되므로 불변
    BASE_COST[:] = BASE_PRICE * (1 - m)
    for lot_list in [WARM[s].lots for s in STORES]:
        pass
    # 로트에 박힌 원가도 갱신
    for s in STORES:
        for lot in WARM[s].lots:
            lot[3] = BASE_PRICE[lot[0]] * (1 - m)
    for s in STORES:
        for d, arr in WARM[s].arrivals.items():
            arr[:, 3] = [BASE_PRICE[int(p)] * (1 - m) for p in arr[:, 0]]

    for bi, beta in enumerate(BETA_GRID2):
        sim.p['beta_disc'] = beta
        el = elasticity_of(beta)
        base = evaluate_fast(no_discount)['profit']
        best_r, best_p = 0.0, base
        for r in DD_GRID2[1:]:
            v = evaluate_fast(dday_policy(r))['profit']
            if v > best_p:
                best_p, best_r = v, r
        MAP[mi, bi] = (best_p / base - 1) * 100 if base > 0 else np.nan
        MAPD[mi, bi] = best_r * 100
        rows.append(dict(마진율=m, 탄력성=el, 최적할인율=best_r,
                         무할인이익=base, 최적이익=best_p,
                         개선율=best_p / base - 1 if base > 0 else np.nan))
    print('  마진 %.0f%% 완료 (%.1f분)' % (m * 100, (time.time() - t0) / 60))

# 원상 복구
sim.p['beta_disc'] = beta_bak
BASE_COST[:] = BASE_COST_ORIG
for s in STORES:
    for lot in WARM[s].lots:
        lot[3] = BASE_COST_ORIG[lot[0]]
    for d, arr in WARM[s].arrivals.items():
        arr[:, 3] = [BASE_COST_ORIG[int(p)] for p in arr[:, 0]]

BOUNDARY = pd.DataFrame(rows)
BOUNDARY.to_csv(f'{OUT}/boundary_map.csv', index=False, encoding='utf-8-sig')
ELAST_AXIS = [elasticity_of(b) for b in BETA_GRID2]

print()
print('할인 도입 시 이익 개선율 (%) — 행=마진율, 열=탄력성')
print(pd.DataFrame(MAP.round(1),
                   index=[f'{m*100:.0f}%' for m in MARGIN_GRID],
                   columns=[f'{e*100:.0f}%' for e in ELAST_AXIS]).to_string())
print()
print('그때의 최적 할인율 (%)')
print(pd.DataFrame(MAPD.astype(int),
                   index=[f'{m*100:.0f}%' for m in MARGIN_GRID],
                   columns=[f'{e*100:.0f}%' for e in ELAST_AXIS]).to_string())
print()
print('소요 %.1f분' % ((time.time() - t0) / 60))

## 6. 정책 비교

임계점은 **룰 기반 마감할인의 이익**이다. 이 값을 넘어야 도입 명분이 생긴다.

In [ ]:
# 임계점 = 현행 대비 개선. 단 현행이 무할인보다 낮으면 무할인이 실질 기준선이 된다.
THRESHOLD = max(R_RULE['profit'], R_NONE['profit'])
BASE_NAME = '룰 기반' if R_RULE['profit'] >= R_NONE['profit'] else '무할인'
print('임계 기준: %s (둘 중 높은 쪽)' % BASE_NAME)

rows = []
for nm, r in [('무할인 (하한선)', R_NONE), ('룰 기반 마감할인 (현행)', R_RULE),
              ('사후 오라클 (상한)', R_ORACLE)]:
    rows.append(dict(정책=nm, 매출=r['revenue'], 이익=r['profit'],
                     폐기수량=r['waste_qty'], 폐기율=r['waste_rate'],
                     마진율=r['margin'],
                     무할인대비=r['profit'] / R_NONE['profit'] - 1,
                     현행대비=r['profit'] / THRESHOLD - 1))
COMPARE = pd.DataFrame(rows)
COMPARE.to_csv(f'{OUT}/policy_comparison.csv', index=False, encoding='utf-8-sig')

print('임계점 (현행 룰 기반 이익): %.0f원' % THRESHOLD)
print()
disp = COMPARE.copy()
for c in ['매출', '이익', '폐기수량']:
    disp[c] = disp[c].map(lambda v: f'{v:,.0f}')
for c in ['폐기율', '마진율', '무할인대비', '현행대비']:
    disp[c] = disp[c].map(lambda v: f'{v:+.2%}' if c.endswith('대비') else f'{v:.2%}')
print(disp.to_string(index=False))

In [ ]:
# 오라클과 현행의 격차 = 후보 생성기(코드3)가 채워야 할 여지
gap = R_ORACLE['profit'] - R_RULE['profit']
print('현행 대비 오라클 여지: %.0f원 (%.2f%%)' % (gap, gap / R_RULE['profit'] * 100))
print('무할인 대비 현행 개선: %.0f원 (%.2f%%)'
      % (R_RULE['profit'] - R_NONE['profit'],
         (R_RULE['profit'] / R_NONE['profit'] - 1) * 100))
print()
print('코드3의 목표: 실시간 정보만으로 이 여지를 얼마나 회수하는가 (후회 최소화)')

## 6.5 후회(regret)와 임계점 마진

In [ ]:
# ── 후회(regret) 정식화 및 임계점 마진
ALPHA = 0.03      # 도입 명분 마진 3% (민감도 분석 대상)
BASE_PROFIT = max(R_RULE['profit'], R_NONE['profit'])
THRESHOLD_A = BASE_PROFIT * (1 + ALPHA) if BASE_PROFIT > 0 else BASE_PROFIT * (1 - ALPHA)

ORACLE_P = R_ORACLE['profit']


def regret(profit):
    """사후 오라클 대비 후회. 0이면 오라클과 동일."""
    return ORACLE_P - profit


def regret_ratio(profit):
    """후회를 무할인~오라클 구간으로 정규화. 1이면 오라클 달성, 0이면 하한선."""
    span = ORACLE_P - R_NONE['profit']
    return (profit - R_NONE['profit']) / span if abs(span) > 1e-9 else np.nan


print('임계점 구성')
print('  기준선(무할인/현행 중 높은 쪽) %12.0f원' % BASE_PROFIT)
print('  마진 alpha = %.0f%%' % (ALPHA * 100))
print('  임계점                        %12.0f원' % THRESHOLD_A)
print()
SPAN = ORACLE_P - R_NONE['profit']
print('후회 지표')
for nm, r in [('무할인', R_NONE), ('룰 기반(현행)', R_RULE), ('사후 오라클', R_ORACLE)]:
    rr = regret_ratio(r['profit'])
    tail = ('달성률 %6.1f%%' % (rr * 100)) if abs(SPAN) > 1e-9 else '달성률 정의불가'
    print('  %-14s 이익 %12.0f · 후회 %12.0f · %s' % (nm, r['profit'], regret(r['profit']), tail))
print()
if abs(SPAN) <= 1e-9:
    print('[주의] 사후 오라클이 무할인과 동일한 이익을 낸다.')
    print('       즉 현재 파라미터에서는 어떤 할인도 이익을 개선하지 못하므로,')
    print('       달성률 구간(무할인 ~ 오라클)이 0이 되어 정의되지 않는다.')
    print('       5.5절 손익분기 분석이 이 상황의 원인을 설명한다.')
else:
    print('코드3의 목표: 실시간 정보만으로 달성률을 100%에 얼마나 근접시키는가')

## 6.6 규모 정합성 — 비율 지표

축소 모형이라 절대 금액은 실제 마트와 직접 비교할 수 없다.
대신 **규모에 무관한 비율 지표**로 상식적 범위 안에 있는지 확인한다.

In [ ]:
# ── 규모 정합성 4종 (비율 지표는 축소 모형에서도 비교 가능)
_dec_days = len(EVAL_ONLY)
_recpt = R_RULE['sold'] / max(_dec_days, 1)     # 일평균 판매수량 (참고)


def ratio_metrics(r, label):
    return dict(
        정책=label,
        신선식품_영업이익률=r['profit'] / max(r['revenue'], 1e-9),
        매출대비_폐기손실=(r['waste_cost'] + r['disposal_fee']) / max(r['revenue'], 1e-9),
        수량기준_폐기율=r['waste_rate'],
        평균_판매단가=r['revenue'] / max(r['sold'], 1e-9),
    )


RATIO = pd.DataFrame([ratio_metrics(R_NONE, '무할인'),
                      ratio_metrics(R_RULE, '룰 기반(현행)'),
                      ratio_metrics(R_ORACLE, '사후 오라클')])
RATIO.to_csv(f'{OUT}/ratio_metrics.csv', index=False, encoding='utf-8-sig')

print('규모 정합성 — 비율 지표 (업계 통상 범위와 대조)')
RATIO_DISP = RATIO.copy()
for c in ['신선식품_영업이익률', '매출대비_폐기손실', '수량기준_폐기율']:
    RATIO_DISP[c] = RATIO_DISP[c].map(lambda v: f'{v:.2%}')
RATIO_DISP['평균_판매단가'] = RATIO_DISP['평균_판매단가'].map(lambda v: f'{v:,.0f}원')
print(RATIO_DISP.to_string(index=False))
print()
print('참고 · 대형마트 신선식품 통상 범위 — 영업이익률 2~5%, 폐기율 1~3%')
print('     (본 시뮬레이션은 3점포 38품목 축소 모형이므로 자릿수 수준 비교만 유효)')

## 7. 판별기 인터페이스 확인

코드3의 후보 생성기가 실제로 붙일 지점을 검증한다.
임의의 후보를 만들어 넣고 승인/반려 판정이 도는지 본다.

In [ ]:
def judge(policy, threshold=None, verbose=True):
    """후보 하나를 평가해 승인/반려 판정"""
    th = THRESHOLD_A if threshold is None else threshold
    r = evaluate_fast(policy)
    ok = r['profit'] > th
    if verbose:
        print('이익 %12.0f원 · 임계 %12.0f원 -> %s' % (r['profit'], th, '승인' if ok else '반려'))
    return ok, r


# 무작위 후보 3개로 인터페이스 점검
t0 = time.time()
for k in range(3):
    tb = {(b, t): float(RNG.choice(GRID)) for b in range(NB) for t in range(NT)}
    ok, r = judge(make_table_policy(tb), verbose=False)
    print('후보 %d · 이익 %12.0f · 기준선대비 %+7.2f%% -> %s'
          % (k + 1, r['profit'], (r['profit'] / BASE_PROFIT - 1) * 100, '승인' if ok else '반려'))
print('\n후보 1개 평가 %.2f초' % ((time.time() - t0) / 3))

## 7.5 판별기 대리모델 — 모델 후보 비교

### 왜 필요한가

판별기 1회 평가에 약 1초가 걸린다. 코드3의 탐색 루프는 에폭마다 여러 후보를 평가하므로,
에폭 100회면 수 분이 소요된다. 롤링 실행은 매시간 돌아야 하므로 이 비용은 그대로 제약이 된다.

그래서 **정책 → 이익을 근사하는 대리모델(surrogate)** 을 학습한다.
탐색 중에는 대리모델로 후보를 걸러내고, 유망한 후보만 실제 판별기로 확인한다.
최적화 분야에서 surrogate-assisted optimization이라 부르는 표준 기법이다.

### 학습 문제 정의

| 항목 | 내용 |
|---|---|
| 입력 X | 정책 벡터 20차원 (카테고리 5 × 잔여기한 4, 각 0~40% 할인율) |
| 출력 y | 판별기가 계산한 12월 총 이익 |
| 표본 | 정책 공간에서 무작위 추출 |

**타깃이 합성 영수증이 아니라 우리 판별기의 출력이라는 점이 중요하다.**
코드1에서 지도학습을 피했던 이유는 정답 라벨이 합성 영수증뿐이라 순환이 생기기 때문이었다.
여기서는 그 문제가 없다. 판별기는 결정론적 시뮬레이터이고,
대리모델의 역할은 그 출력을 빠르게 근사하는 것뿐이다.

### 후보 모델과 평가 지표

| 후보 | 계열 |
|---|---|
| Ridge 선형회귀 | 선형 기준선 |
| 결정트리 | 단일 트리 |
| 랜덤 포레스트 | 배깅 앙상블 |
| LightGBM | 부스팅 앙상블 |

| 지표 | 보는 것 |
|---|---|
| RMSE | 이익 예측 오차의 크기 |
| R² | 설명력 |
| **Spearman 순위상관** | 후보의 **순서**를 얼마나 정확히 매기는가 |

세 번째 지표를 넣은 이유가 있다. 탐색에서 실제로 필요한 것은 이익의 절대값이 아니라
**어느 후보가 더 나은가**이다. RMSE가 다소 높아도 순위를 정확히 매기면 탐색에는 충분하다.
따라서 순위상관을 1순위 지표로 둔다.

평가는 **5-fold 교차검증**으로 한다.

In [ ]:
# ── 학습 데이터 생성: 정책 공간에서 무작위 표본 추출
from sklearn.model_selection import KFold
from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.metrics import roc_auc_score, precision_recall_curve, roc_curve
from scipy.stats import spearmanr

# LightGBM — 코랩에는 기본 설치되어 있다. 없으면 설치하고, 그래도 실패하면
# scikit-learn의 히스토그램 부스팅으로 대체한다(동일 계열, 결과 해석 동일).
LGB_OK = True
try:
    from lightgbm import LGBMRegressor, LGBMClassifier
except ImportError:
    try:
        import subprocess
        subprocess.run(['pip', 'install', '-q', 'lightgbm'], check=True, capture_output=True)
        from lightgbm import LGBMRegressor, LGBMClassifier
    except Exception:
        LGB_OK = False
        from sklearn.ensemble import HistGradientBoostingRegressor, HistGradientBoostingClassifier
print('부스팅 구현:', 'LightGBM' if LGB_OK else 'sklearn HistGradientBoosting (대체)')

N_SAMPLE = 160
FEAT_NAMES = COORD_LABEL                                      # 12차원 (잔여기한 x 시간대)

t0 = time.time()
Xs, ys = [], []
for k in range(N_SAMPLE):
    vec = RNG.choice(GRID, size=len(COORDS))
    tb = {COORDS[j]: float(vec[j]) for j in range(len(COORDS))}
    r = evaluate_fast(make_table_policy(tb))
    Xs.append(vec); ys.append(r['profit'])
    if (k + 1) % 40 == 0:
        print('  %3d/%d  (%.1f분 경과)' % (k + 1, N_SAMPLE, (time.time() - t0) / 60))

X = np.array(Xs, dtype=float)
y = np.array(ys, dtype=float)
X_SUR, Y_SUR = X, y
print('\n표본 %d개 생성 · %.1f분' % (len(y), (time.time() - t0) / 60))
print('이익 범위 %.0f ~ %.0f원 (표준편차 %.0f)' % (y.min(), y.max(), y.std()))

In [ ]:
# ── 후보 4종 · 5-fold 교차검증
BOOST_NAME = 'LightGBM' if LGB_OK else 'Hist 그래디언트 부스팅'


def make_models():
    return {
        'Ridge 선형회귀': Ridge(alpha=1.0),
        '결정트리': DecisionTreeRegressor(max_depth=6, random_state=42),
        '랜덤 포레스트': RandomForestRegressor(n_estimators=300, max_depth=10,
                                          random_state=42, n_jobs=-1),
        BOOST_NAME: (LGBMRegressor(n_estimators=400, learning_rate=0.06, num_leaves=15,
                                   min_child_samples=8, random_state=42, verbose=-1)
                     if LGB_OK else
                     HistGradientBoostingRegressor(max_iter=400, learning_rate=0.06,
                                                   max_leaf_nodes=15, min_samples_leaf=8,
                                                   random_state=42)),
    }


kf = KFold(n_splits=5, shuffle=True, random_state=42)
rows, PRED = [], {}
for name in make_models():
    oof = np.zeros(len(y))
    t0 = time.time()
    for tr, te in kf.split(X):
        m = make_models()[name]
        m.fit(X[tr], y[tr])
        oof[te] = m.predict(X[te])
    PRED[name] = oof
    rows.append(dict(
        모델=name,
        RMSE=float(np.sqrt(mean_squared_error(y, oof))),
        MAE=float(mean_absolute_error(y, oof)),
        R2=float(r2_score(y, oof)),
        Spearman순위상관=float(spearmanr(y, oof).statistic),
        학습시간초=time.time() - t0,
    ))

SURROGATE = pd.DataFrame(rows).sort_values('Spearman순위상관', ascending=False).reset_index(drop=True)
SURROGATE.to_csv(f'{OUT}/surrogate_benchmark.csv', index=False, encoding='utf-8-sig')

_d = SURROGATE.copy()
_d['RMSE'] = _d.RMSE.map(lambda v: f'{v:,.0f}')
_d['MAE'] = _d.MAE.map(lambda v: f'{v:,.0f}')
_d['R2'] = _d.R2.map(lambda v: f'{v:.4f}')
_d['Spearman순위상관'] = _d.Spearman순위상관.map(lambda v: f'{v:.4f}')
_d['학습시간초'] = _d.학습시간초.map(lambda v: f'{v:.2f}')
print('대리모델 후보 비교 (5-fold 교차검증)')
print(_d.to_string(index=False))

BEST_SUR = SURROGATE.iloc[0].모델
print('\n채택: %s' % BEST_SUR)

In [ ]:
# ── 보조 트랙: 승인/반려 이진분류 (임계점 통과 여부)
# 임계점 기준 통과 후보 수 (참고)
n_pass = int((y > THRESHOLD_A).sum())
print('임계점(%.0f원) 통과 후보: %d / %d' % (THRESHOLD_A, n_pass, len(y)))
if n_pass == 0:
    print('  → 무작위 후보 중 임계점을 넘는 것이 하나도 없다.')
    print('    5.5절 손익분기 분석과 일관된 결과다 (현재 탄력성에서는 할인이 이익을 못 낸다).')

# 탐색에서 대리모델의 실제 역할은 「유망 후보 걸러내기」다.
# 따라서 분류 타깃을 상위 25% 후보 여부로 둔다.
CUT = float(np.quantile(y, 0.75))
y_cls = (y >= CUT).astype(int)
print()
print('분류 타깃: 상위 25%% 후보 (이익 %.0f원 이상) · 양성 비율 %.1f%%'
      % (CUT, y_cls.mean() * 100))

if 0 < y_cls.mean() < 1:
    from sklearn.linear_model import LogisticRegression
    from sklearn.ensemble import RandomForestClassifier
    clf_models = {
        '로지스틱 회귀': LogisticRegression(max_iter=2000),
        '랜덤 포레스트': RandomForestClassifier(n_estimators=300, max_depth=8,
                                            random_state=42, n_jobs=-1),
        BOOST_NAME: (LGBMClassifier(n_estimators=300, learning_rate=0.06, num_leaves=15,
                                    min_child_samples=8, random_state=42, verbose=-1)
                     if LGB_OK else
                     HistGradientBoostingClassifier(max_iter=300, learning_rate=0.06,
                                                    max_leaf_nodes=15, min_samples_leaf=8,
                                                    random_state=42)),
    }
    crows, CPRED = [], {}
    for nm, mdl in clf_models.items():
        oof = np.zeros(len(y_cls))
        for tr, te in kf.split(X):
            m = type(mdl)(**mdl.get_params())
            m.fit(X[tr], y_cls[tr])
            oof[te] = m.predict_proba(X[te])[:, 1]
        CPRED[nm] = oof
        auc = roc_auc_score(y_cls, oof)
        pr, rc, th = precision_recall_curve(y_cls, oof)
        f1 = 2 * pr * rc / np.maximum(pr + rc, 1e-9)
        bi = int(np.nanargmax(f1))
        crows.append(dict(모델=nm, ROC_AUC=auc,
                          최적임계값=float(th[min(bi, len(th) - 1)]),
                          정밀도=float(pr[bi]), 재현율=float(rc[bi]), F1=float(f1[bi])))
    CLASSIFY = pd.DataFrame(crows).sort_values('ROC_AUC', ascending=False).reset_index(drop=True)
    CLASSIFY.to_csv(f'{OUT}/approval_classifier.csv', index=False, encoding='utf-8-sig')
    print()
    print('승인/반려 분류 성능 (5-fold)')
    print(CLASSIFY.round(4).to_string(index=False))
else:
    CLASSIFY, CPRED = None, None
    print('한쪽 클래스만 존재하여 분류 트랙 생략 (임계점을 넘는 후보가 없거나 전부 넘음)')

In [ ]:
# ── 대리모델의 실제 효용: 속도 이득
best_model = make_models()[BEST_SUR]
best_model.fit(X, y)

t0 = time.time()
for _ in range(2000):
    best_model.predict(RNG.choice(GRID, size=(1, len(COORDS))).astype(float))
t_sur = (time.time() - t0) / 2000

print('판별기 1회      %8.2f ms' % (t_fast * 1000))
print('대리모델 1회    %8.3f ms' % (t_sur * 1000))
print('속도 이득       %8.0f 배' % (t_fast / max(t_sur, 1e-12)))
print()
print('코드3 탐색에서 대리모델로 후보를 거르고 상위 후보만 판별기로 확인하면,')
print('같은 시간에 훨씬 많은 후보를 탐색할 수 있다.')

# 상위 후보 선별 정확도: 대리모델 상위 10개가 실제로 얼마나 좋은가
oof_best = PRED[BEST_SUR]
top_pred = np.argsort(-oof_best)[:10]
top_true = np.argsort(-y)[:10]
hit = len(set(top_pred) & set(top_true))
print()
print('상위 10개 후보 선별 정확도: %d/10 일치' % hit)
print('대리모델 상위 10개의 실제 이익 평균 %.0f원 (전체 평균 %.0f원, 실제 상위 10개 %.0f원)'
      % (y[top_pred].mean(), y.mean(), y[top_true].mean()))

## 8. 시각화

In [ ]:
import subprocess, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib import font_manager

_ok = False
try:
    import koreanize_matplotlib; _ok = True
except ImportError:
    try:
        subprocess.run(['pip', 'install', '-q', 'koreanize-matplotlib'], check=True, capture_output=True)
        import koreanize_matplotlib; _ok = True
    except Exception:
        pass
if not _ok:
    try:
        subprocess.run(['apt-get', '-qq', 'install', '-y', 'fonts-nanum'], check=True, capture_output=True)
        for fp_ in font_manager.findSystemFonts(fontpaths=['/usr/share/fonts/truetype/nanum']):
            font_manager.fontManager.addfont(fp_)
        plt.rcParams['font.family'] = 'NanumGothic'; _ok = True
    except Exception as e:
        print('한글 폰트 설정 실패:', e)
print('한글 폰트 준비:', _ok)

plt.rcParams.update({'figure.dpi': 140, 'savefig.dpi': 200, 'axes.unicode_minus': False,
                     'font.size': 11, 'axes.titlesize': 13, 'axes.titleweight': 'bold',
                     'axes.grid': True, 'grid.alpha': 0.25, 'axes.spines.top': False,
                     'axes.spines.right': False, 'figure.facecolor': 'white'})
FIG = f'{OUT}/figs'; os.makedirs(FIG, exist_ok=True)
NAVY, BLUE, GREEN, ORANGE, GRAY, RED = '#1F3864', '#2E75B6', '#2E9E5B', '#ED9B40', '#9AA5B1', '#D64545'

In [ ]:
labels = ['무할인\n(하한선)', '룰 기반 마감할인\n(현행)', '사후 오라클\n(상한)']
prof = [R_NONE['profit'], R_RULE['profit'], R_ORACLE['profit']]
wst = [R_NONE['waste_rate'] * 100, R_RULE['waste_rate'] * 100, R_ORACLE['waste_rate'] * 100]
cols = [GRAY, BLUE, GREEN]

fig, ax = plt.subplots(1, 3, figsize=(15, 4.4))
b = ax[0].bar(labels, np.array(prof) / 1e6, color=cols)
ax[0].set_ylabel('이익 (백만원)'); ax[0].set_title('정책별 이익')
for r, v in zip(b, prof):
    ax[0].text(r.get_x() + r.get_width()/2, r.get_height(), f'{v/1e6:.1f}M',
               ha='center', va='bottom', fontsize=10, fontweight='bold')
ax[0].axhline(R_RULE['profit']/1e6, ls='--', color=RED, lw=1.3)

b = ax[1].bar(labels, wst, color=cols)
ax[1].set_ylabel('폐기율 (%)'); ax[1].set_title('정책별 폐기율')
for r, v in zip(b, wst):
    ax[1].text(r.get_x() + r.get_width()/2, r.get_height(), f'{v:.2f}%',
               ha='center', va='bottom', fontsize=10, fontweight='bold')

imp = [0, (R_RULE['profit']/R_NONE['profit']-1)*100, (R_ORACLE['profit']/R_NONE['profit']-1)*100]
b = ax[2].bar(labels, imp, color=cols)
ax[2].set_ylabel('무할인 대비 이익 개선 (%)'); ax[2].set_title('개선 폭')
for r, v in zip(b, imp):
    ax[2].text(r.get_x() + r.get_width()/2, r.get_height(), f'{v:+.2f}%',
               ha='center', va='bottom', fontsize=10, fontweight='bold')

fig.suptitle('정책 비교 — 12월 3개 점포 · 38개 신선식품',
             fontsize=15, fontweight='bold', y=1.03)
fig.tight_layout(); fig.savefig(f'{FIG}/fig1_policy_comparison.png', bbox_inches='tight'); plt.close(fig)
print('fig1 저장')

In [ ]:
# [2] 오라클 최적 할인 테이블 히트맵 (잔여기한 x 시간대)
M = ORACLE_DF.values * 100
fig, ax = plt.subplots(figsize=(7.8, 4.2))
im = ax.imshow(M, cmap='YlOrRd', vmin=0, vmax=40, aspect='auto')
ax.set_xticks(range(NT)); ax.set_xticklabels(TIME_LABEL, fontsize=10)
ax.set_yticks(range(NB)); ax.set_yticklabels(DTE_LABEL)
for i in range(NB):
    for j in range(NT):
        ax.text(j, i, f'{M[i,j]:.0f}%', ha='center', va='center',
                color='white' if M[i, j] > 22 else '#333', fontweight='bold')
ax.set_title('사후 오라클이 찾은 최적 할인율 (잔여기한 × 시간대)')
ax.grid(False)
fig.colorbar(im, ax=ax, label='할인율 (%)')
fig.tight_layout(); fig.savefig(f'{FIG}/fig2_oracle_table.png', bbox_inches='tight'); plt.close(fig)

# 이익 분해
fig, ax = plt.subplots(figsize=(11, 4.4))
x = np.arange(3); w = 0.2
vals = {
    '매출': [R_NONE['revenue'], R_RULE['revenue'], R_ORACLE['revenue']],
    '매출원가': [-R_NONE['cogs'], -R_RULE['cogs'], -R_ORACLE['cogs']],
    '폐기손실': [-R_NONE['waste_cost'], -R_RULE['waste_cost'], -R_ORACLE['waste_cost']],
    '폐기처리비': [-R_NONE['disposal_fee'], -R_RULE['disposal_fee'], -R_ORACLE['disposal_fee']],
}
for i, (k, v) in enumerate(vals.items()):
    ax.bar(x + (i - 1.5) * w, np.array(v) / 1e6, w, label=k)
ax.set_xticks(x); ax.set_xticklabels(['무할인', '룰 기반', '오라클'])
ax.set_ylabel('금액 (백만원)'); ax.set_title('이익 구성 분해'); ax.legend(ncol=4)
ax.axhline(0, color='#333', lw=0.8)
fig.tight_layout(); fig.savefig(f'{FIG}/fig3_profit_decomposition.png', bbox_inches='tight'); plt.close(fig)
print('fig2, fig3 저장')

In [ ]:
# [4] 손익분기 탄력성 — 이 프로젝트의 핵심 그래프
fig, ax = plt.subplots(1, 2, figsize=(13, 4.4))
bx = BREAKEVEN.탄력성_10pp * 100
by = BREAKEVEN.개선율 * 100
ax[0].plot(bx, by, 'o-', lw=2.5, color=BLUE, ms=7)
ax[0].axhline(0, color='#333', lw=1)
if not np.isnan(BE_ELAST):
    ax[0].axvline(BE_ELAST * 100, ls='--', color=RED, lw=1.6)
    ax[0].text(BE_ELAST * 100 + 6, max(by) * 0.55,
               '손익분기 %.1f%%/10%%p' % (BE_ELAST * 100),
               color=RED, fontsize=10, fontweight='bold')
ax[0].axvline(CUR_ELAST * 100, ls='--', color=GREEN, lw=1.6)
ax[0].text(CUR_ELAST * 100 + 4, max(by) * 0.85,
           '현재 가정 %.1f%%/10%%p' % (CUR_ELAST * 100),
           color=GREEN, fontsize=10, fontweight='bold')
ax[0].set_xlabel('가격 탄력성 (할인 10%p당 판매량 증가율, %)')
ax[0].set_ylabel('무할인 대비 이익 개선 (%)')
ax[0].set_title('할인이 이익이 되는 조건')

b = ax[1].bar(range(len(BREAKEVEN)), BREAKEVEN.최적_D_day할인율 * 100,
              color=[GRAY if v == 0 else GREEN for v in BREAKEVEN.최적_D_day할인율])
ax[1].set_xticks(range(len(BREAKEVEN)))
ax[1].set_xticklabels(['%.0f%%' % (v * 100) for v in BREAKEVEN.탄력성_10pp], rotation=20)
ax[1].set_xlabel('가격 탄력성 (%/10%p)')
ax[1].set_ylabel('최적 D-day 할인율 (%)')
ax[1].set_title('탄력성별 최적 마감 할인율')
for r, v in zip(b, BREAKEVEN.최적_D_day할인율):
    ax[1].text(r.get_x() + r.get_width() / 2, r.get_height(), '%.0f%%' % (v * 100),
               ha='center', va='bottom', fontsize=10, fontweight='bold')

fig.suptitle('손익분기 탄력성 분석 — 결론은 데이터가 아니라 가정에서 나온다',
             fontsize=15, fontweight='bold', y=1.03)
fig.tight_layout(); fig.savefig(f'{FIG}/fig4_breakeven.png', bbox_inches='tight'); plt.close(fig)
print('fig4 저장')

In [ ]:
# [5] 대리모델 후보 비교
fig, ax = plt.subplots(1, 3, figsize=(15, 4.3))
names = SURROGATE.모델.tolist()
xs = np.arange(len(names))
cols = [GREEN if n == BEST_SUR else GRAY for n in names]

ax[0].bar(xs, SURROGATE.Spearman순위상관, color=cols)
ax[0].set_xticks(xs); ax[0].set_xticklabels(names, rotation=18, ha='right')
ax[0].set_ylim(0, 1.05); ax[0].set_title('순위상관 (높을수록 좋음) · 1순위 지표')
for i, v in enumerate(SURROGATE.Spearman순위상관):
    ax[0].text(i, v, f'{v:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax[1].bar(xs, SURROGATE.R2, color=cols)
ax[1].set_xticks(xs); ax[1].set_xticklabels(names, rotation=18, ha='right')
ax[1].set_title('R² (설명력)')
for i, v in enumerate(SURROGATE.R2):
    ax[1].text(i, v, f'{v:.3f}', ha='center', va='bottom', fontsize=10)

ax[2].bar(xs, SURROGATE.RMSE / 1e6, color=cols)
ax[2].set_xticks(xs); ax[2].set_xticklabels(names, rotation=18, ha='right')
ax[2].set_ylabel('백만원'); ax[2].set_title('RMSE (낮을수록 좋음)')
for i, v in enumerate(SURROGATE.RMSE):
    ax[2].text(i, v / 1e6, f'{v/1e6:.1f}', ha='center', va='bottom', fontsize=10)

fig.suptitle(f'판별기 대리모델 후보 비교 — 5-fold 교차검증 · 채택 {BEST_SUR}',
             fontsize=15, fontweight='bold', y=1.03)
fig.tight_layout(); fig.savefig(f'{FIG}/fig5_surrogate.png', bbox_inches='tight'); plt.close(fig)

# [6] 예측 대 실제 산점도 + ROC
ncol = 2 if CLASSIFY is not None else 1
fig, ax = plt.subplots(1, ncol, figsize=(6.5 * ncol, 4.4))
ax = np.atleast_1d(ax)
ax[0].scatter(Y_SUR / 1e6, PRED[BEST_SUR] / 1e6, s=18, alpha=0.6, color=BLUE)
lim = [min(Y_SUR.min(), PRED[BEST_SUR].min()) / 1e6, max(Y_SUR.max(), PRED[BEST_SUR].max()) / 1e6]
ax[0].plot(lim, lim, ls='--', color=RED, lw=1.3)
ax[0].set_xlabel('실제 이익 (백만원)'); ax[0].set_ylabel('대리모델 예측 (백만원)')
ax[0].set_title(f'{BEST_SUR} — 예측 대 실제 (교차검증)')

if CLASSIFY is not None:
    for nm in CPRED:
        fpr, tpr, _ = roc_curve(y_cls, CPRED[nm])
        a_ = roc_auc_score(y_cls, CPRED[nm])
        ax[1].plot(fpr, tpr, lw=2, label=f'{nm} (AUC={a_:.3f})')
    ax[1].plot([0, 1], [0, 1], ls='--', color=GRAY, lw=1)
    ax[1].set_xlabel('위양성률'); ax[1].set_ylabel('재현율')
    ax[1].set_title('승인/반려 분류 ROC'); ax[1].legend(fontsize=9)

fig.tight_layout(); fig.savefig(f'{FIG}/fig6_surrogate_detail.png', bbox_inches='tight'); plt.close(fig)
print('fig5, fig6 저장')

In [ ]:
# [7] 경계 지도
fig, ax = plt.subplots(1, 2, figsize=(13, 4.6))
xt = [f'{e*100:.0f}%' for e in ELAST_AXIS]
yt = [f'{m*100:.0f}%' for m in MARGIN_GRID]

im0 = ax[0].imshow(MAP, cmap='YlGn', aspect='auto',
                   vmin=0, vmax=max(1, np.nanmax(MAP)))
ax[0].set_xticks(range(len(xt))); ax[0].set_xticklabels(xt)
ax[0].set_yticks(range(len(yt))); ax[0].set_yticklabels(yt)
ax[0].set_xlabel('가격 탄력성 (할인 10%p당 판매 증가율)')
ax[0].set_ylabel('정가 마진율')
ax[0].set_title('할인 도입 시 이익 개선율 (%) — 0이면 할인 무의미')
for i in range(MAP.shape[0]):
    for j in range(MAP.shape[1]):
        ax[0].text(j, i, f'{MAP[i,j]:+.1f}', ha='center', va='center',
                   fontsize=11, fontweight='bold',
                   color='white' if MAP[i, j] > np.nanmax(MAP) * 0.6 else '#222')
ax[0].grid(False)
ax[0].add_patch(plt.Rectangle((-0.5, -0.5), 1, 1, fill=False, ec=NAVY, lw=3))
ax[0].text(-0.42, -0.72, '현재 위치', color=NAVY, fontsize=10, fontweight='bold')

im1 = ax[1].imshow(MAPD, cmap='YlOrRd', aspect='auto', vmin=0, vmax=40)
ax[1].set_xticks(range(len(xt))); ax[1].set_xticklabels(xt)
ax[1].set_yticks(range(len(yt))); ax[1].set_yticklabels(yt)
ax[1].set_xlabel('가격 탄력성 (할인 10%p당 판매 증가율)')
ax[1].set_ylabel('정가 마진율')
ax[1].set_title('판별기가 선택한 최적 할인율 (%)')
for i in range(MAPD.shape[0]):
    for j in range(MAPD.shape[1]):
        ax[1].text(j, i, f'{MAPD[i,j]:.0f}%', ha='center', va='center',
                   fontsize=11, fontweight='bold',
                   color='white' if MAPD[i, j] > 22 else '#333')
ax[1].grid(False)

fig.suptitle('경계 지도 — 어떤 조건에서 할인이 이익인가',
             fontsize=15, fontweight='bold', y=1.03)
fig.tight_layout(); fig.savefig(f'{FIG}/fig7_boundary.png', bbox_inches='tight'); plt.close(fig)
print('fig7 저장')

## 9. 결과 묶어서 내려받기

In [ ]:
import zipfile, datetime, platform
NL = chr(10)


def _md(df):
    try:
        return df.to_markdown(index=False)
    except Exception:
        return '```' + NL + df.to_string(index=False) + NL + '```'


L = []
L.append('# 코드2 (판별기) 실행 결과')
L.append('')
L.append('실행 시각: ' + datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S'))
L.append('평가 구간: ' + str(EVAL_START.date()) + ' ~ ' + str(EVAL_END.date())
         + ' · 점포 ' + str(len(STORES)) + '개 · 상품 ' + str(P) + '개')
L.append('예열 구간: ' + str(WARMUP_START.date()) + ' ~ '
         + str((EVAL_START - pd.Timedelta(days=1)).date()) + ' (룰 기반으로 재고 상태 형성)')
L.append('')
L.append('## 1. 정책 비교')
L.append('')
L.append(_md(disp))
L.append('')
L.append('실질 기준선: ' + BASE_NAME + ' ' + format(BASE_PROFIT, ',.0f') + '원')
L.append('')
L.append('- 무할인 대비 현행 개선: ' + format(R_RULE['profit'] / R_NONE['profit'] - 1, '+.2%'))
L.append('- 오라클 대비 현행의 손실: ' + format(R_RULE['profit'] - R_ORACLE['profit'], ',.0f') + '원')
L.append('- 기준선(' + BASE_NAME + ') 대비 오라클 여지: '
         + format(R_ORACLE['profit'] - BASE_PROFIT, ',.0f') + '원')
L.append('')
L.append('## 2. 손익분기 탄력성 (핵심 결과)')
L.append('')
L.append(_md(BREAKEVEN.round(4)))
L.append('')
L.append('- 현재 가정 탄력성: ' + format(CUR_ELAST, '+.2%') + ' / 10%p')
L.append('- 할인이 이익이 되는 손익분기: '
         + (format(BE_ELAST, '+.1%') + ' / 10%p' if not np.isnan(BE_ELAST) else '탐색 범위 내 없음'))
L.append('- 정가 마진율 ' + format(float((1 - BASE_COST / BASE_PRICE).mean()), '.1%')
         + ' 에서는 할인 10%p가 단위 마진의 절반 이상을 소모한다')
L.append('')
L.append('해석 — "할인하지 말라"는 결론은 데이터가 아니라 탄력성 가정에서 나온다.')
L.append('실측 탄력성이 손익분기를 넘는지가 도입 판단의 핵심이며, 이는 실증(A/B 테스트) 대상이다.')
L.append('')
L.append('## 2.5 경계 지도 (마진율 x 탄력성)')
L.append('')
L.append('할인 도입 시 이익 개선율 (%) — 행=마진율, 열=탄력성')
L.append('')
L.append(_md(pd.DataFrame(MAP.round(1),
            index=[format(m_, '.0%') for m_ in MARGIN_GRID],
            columns=[format(e_, '.0%') for e_ in ELAST_AXIS]).reset_index()
            .rename(columns={'index': '마진율'})))
L.append('')
L.append('그때의 최적 할인율 (%)')
L.append('')
L.append(_md(pd.DataFrame(MAPD.astype(int),
            index=[format(m_, '.0%') for m_ in MARGIN_GRID],
            columns=[format(e_, '.0%') for e_ in ELAST_AXIS]).reset_index()
            .rename(columns={'index': '마진율'})))
L.append('')
L.append('판별기의 역할은 「할인하라/하지 말라」가 아니라 이 경계를 찾는 것이다.')
L.append('')

L.append('## 3. 판별기 타당성 검증')
L.append('')
L.append('현행 룰 기반 정책 하에서 원천 실적 재현도')
L.append('')
L.append('| 지표 | 원천 | 시뮬 | 오차 |')
L.append('|---|---|---|---|')
L.append('| 12월 판매수량 | ' + format(TGT_TOTAL, ',.0f') + ' | ' + format(R_RULE['sold'], ',.0f')
         + ' | ' + format(R_RULE['sold'] / TGT_TOTAL - 1, '+.2%') + ' |')
L.append('| 폐기율 | ' + format(SRC_WASTE_RATE, '.2%') + ' | ' + format(R_RULE['waste_rate'], '.2%')
         + ' | ' + format(R_RULE['waste_rate'] - SRC_WASTE_RATE, '+.2%') + ' |')
L.append('')
L.append('## 4. 후회(regret)와 임계점')
L.append('')
L.append('| 정책 | 이익 | 후회 | 달성률 |')
L.append('|---|---|---|---|')
for _nm, _r in [('무할인', R_NONE), ('룰 기반(현행)', R_RULE), ('사후 오라클', R_ORACLE)]:
    L.append('| ' + _nm + ' | ' + format(_r['profit'], ',.0f') + ' | '
             + format(regret(_r['profit']), ',.0f') + ' | '
             + (format(regret_ratio(_r['profit']), '.1%') if abs(SPAN) > 1e-9 else '정의불가') + ' |')
L.append('')
L.append('임계점 = 기준선(' + BASE_NAME + ') ' + format(BASE_PROFIT, ',.0f')
         + '원 x (1 + alpha ' + format(ALPHA, '.0%') + ') = ' + format(THRESHOLD_A, ',.0f') + '원')
if abs(SPAN) <= 1e-9:
    L.append('')
    L.append('사후 오라클이 무할인과 동일한 이익을 내므로 달성률 구간이 0이 되어 정의되지 않는다.')
    L.append('현재 파라미터에서는 어떤 할인도 이익을 개선하지 못한다는 뜻이며, 2절이 그 원인을 설명한다.')
L.append('')

L.append('## 5. 규모 정합성 (비율 지표)')
L.append('')
L.append(_md(RATIO_DISP))
L.append('')

L.append('## 6. 대리모델 후보 비교 (5-fold 교차검증)')
L.append('')
L.append(_md(SURROGATE.round(4)))
L.append('')
L.append('채택: **' + BEST_SUR + '** — 순위상관 기준')
L.append('')
if CLASSIFY is not None:
    L.append('승인/반려 분류 성능')
    L.append('')
    L.append(_md(CLASSIFY.round(4)))
    L.append('')

L.append('## 7. 사후 오라클 최적 할인 테이블')
L.append('')
L.append(_md(((ORACLE_DF * 100).astype(int).astype(str) + '%').reset_index()
             .rename(columns={'index': '잔여기한'})))
L.append('')
L.append('## 8. 이익 구성')
L.append('')
L.append('| 항목 | 무할인 | 룰 기반 | 오라클 |')
L.append('|---|---|---|---|')
for k, lab in [('revenue', '매출'), ('cogs', '매출원가'), ('waste_cost', '폐기손실'),
               ('disposal_fee', '폐기처리비'), ('profit', '이익'),
               ('sold', '판매수량'), ('waste_qty', '폐기수량')]:
    L.append('| ' + lab + ' | ' + format(R_NONE[k], ',.0f') + ' | '
             + format(R_RULE[k], ',.0f') + ' | ' + format(R_ORACLE[k], ',.0f') + ' |')
L.append('')
L.append('## 9. 판별기 성능')
L.append('')
L.append('후보 1개 평가에 약 ' + format((time.time() - t0) / 3, '.2f') + '초 (12월 31일 x 3점포)')

summary = NL.join(L)
with open(f'{OUT}/run_summary_nb2.md', 'w', encoding='utf-8') as f:
    f.write(summary)
print(summary[:1800])

In [ ]:
ZIP = 'nb2_results.zip'
with zipfile.ZipFile(ZIP, 'w', zipfile.ZIP_DEFLATED) as z:
    for root, _d, fns in os.walk(OUT):
        for fn in sorted(fns):
            fp = os.path.join(root, fn)
            arc = os.path.relpath(fp, OUT)
            z.write(fp, arcname=arc)
            print('  담음:', arc, '(' + str(round(os.path.getsize(fp) / 1024, 1)) + ' KB)')
print()
print(ZIP, '생성 완료 —', round(os.path.getsize(ZIP) / 1024, 1), 'KB')
try:
    from google.colab import files
    files.download(ZIP)
except Exception:
    print('로컬 환경 — 자동 다운로드 생략:', os.path.abspath(ZIP))

## 10. 다음 단계

판별기가 완성됐다. 코드3에서 할 일은 하나다.

**`policy(store_id, date, hour, min_dte, avail) -> (38,) 할인율` 함수를 더 좋게 만드는 것.**

판별기는 정책의 내부를 알 필요가 없다. 함수만 주면 이익을 돌려준다.

### 코드3 계획

1. **후보 생성기** — 확률분포에서 정책을 샘플링, 제약(단조 비증가·원가 하한·40% 상한) 내장
2. **탐색 루프** — CEM(교차 엔트로피법)으로 상위 성과 후보 쪽으로 분포 갱신
3. **롤링 재계획** — 매 시각 폐점까지 재계획하고 당해 시각만 실행
4. **후회(regret) 측정** — 사후 오라클 대비 얼마나 근접했는가

### 알려진 한계

- 오라클은 카테고리 × 잔여기한 격자에서만 탐색한 상한이다. 진짜 최적해는 더 높을 수 있다.
- 감모율 2.55%를 전 상품 동일하게 적용했다. 원천데이터 실측 평균이다.
- 판매수량을 기대값(연속값)으로 계산한다. 몬테카를로를 쓰지 않아 분산 정보는 없다.